# S1 — Boosting, arrêt précoce et monotonie (génie civil)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic2/notebooks/IC2_S1_boosting_monotonie.ipynb)
> Données : dossier `IC2_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn, torch, shap.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['beton_8_constituants.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic2/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import PartialDependenceDisplay
b = pd.read_csv('beton_8_constituants.csv'); X, y = b.drop(columns='resistance_MPa'), b['resistance_MPa']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=1)

## 1. Taux d’apprentissage

In [ ]:
for lr in [0.3, 0.1, 0.03]:
    m = HistGradientBoostingRegressor(max_iter=800, learning_rate=lr, early_stopping=False, random_state=0).fit(X_tr, y_tr)
    r = [mean_squared_error(y_te, p)**0.5 for p in m.staged_predict(X_te)]
    plt.plot(r, label=f'lr={lr}'); print(lr, round(min(r), 2), int(np.argmin(r)) + 1)
plt.ylim(4, 12); plt.legend(); plt.show()

## 2. Arrêt précoce

In [ ]:
gb = HistGradientBoostingRegressor(max_iter=1500, learning_rate=0.05, early_stopping=True, validation_fraction=0.2, n_iter_no_change=50, random_state=0).fit(X_tr, y_tr)
print('arbres :', gb.n_iter_, ' R2 test :', round(r2_score(y_te, gb.predict(X_te)), 3))

## 3. Monotonie avec 110 essais seulement

In [ ]:
mono = [1 if c in ('ciment_kg', 'age_j', 'laitier_kg') else -1 if c == 'eau_kg' else 0 for c in X.columns]
petit = X_tr.sample(110, random_state=3); yp = y_tr[petit.index]
m0 = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, min_samples_leaf=5, random_state=0).fit(petit, yp)
m1 = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, min_samples_leaf=5, monotonic_cst=mono, random_state=0).fit(petit, yp)
print('sans :', round(r2_score(y_te, m0.predict(X_te)), 3), ' avec :', round(r2_score(y_te, m1.predict(X_te)), 3))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
PartialDependenceDisplay.from_estimator(m0, petit, ['eau_kg'], ax=ax[0]); PartialDependenceDisplay.from_estimator(m1, petit, ['eau_kg'], ax=ax[1]); plt.show()

**Question :** avec toutes les éprouvettes, l’effet des contraintes est-il encore visible ? Pourquoi ?